# Step 10: Regression Multi-Horizon Prediction
## Predict actual Vt values (continuous target) at multiple process completion points

This notebook trains CatBoostRegressor models at different process completion points to predict the actual threshold voltage (Vt) value instead of binary outlier classification.

**Key differences from classification:**
- Target: Continuous Vt value (not binary outlier)
- Model: CatBoostRegressor (not CatBoostClassifier)
- Metrics: RMSE, MAE, R² (not ROC AUC, PR AUC, F1)
- Configurable Vt type: NFET1_VT, PFET2_VT, NFET2_VT, or PFET1_VT

## 1. Setup and Configuration

In [23]:
import time
import json
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

start_time = time.time()
print("=" * 60)
print("Step 10: Regression Multi-Horizon Prediction")
print("=" * 60)

Step 10: Regression Multi-Horizon Prediction


In [24]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current directory: {Path.cwd()}")
print(f"\nData directory exists: {(Path.cwd() / 'data').exists()}")
print(f"Outputs directory exists: {(Path.cwd() / 'outputs').exists()}")

Project root: d:\capstone_pipeline
Current directory: d:\capstone_pipeline

Data directory exists: True
Outputs directory exists: True


In [25]:
# CONFIGURATION
vt_type = 'NFET1_VT'  # Options: NFET1_VT, PFET2_VT, NFET2_VT, PFET1_VT
force = False  # Set to True to force rebuild

print(f"\n[CONFIG] Vt type: {vt_type}")
print(f"[CONFIG] Force rebuild: {force}")


[CONFIG] Vt type: NFET1_VT
[CONFIG] Force rebuild: False


In [26]:
# Setup paths
data_dir = Path("data")
output_dir = Path("outputs")
features_dir = output_dir / "features"
models_dir = output_dir / "models"
plots_dir = output_dir / "plots"
models_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

sentinel = models_dir / "10_regression_horizons.done"
if sentinel.exists() and not force:
    print(f"\n[OK] Regression horizon models already trained (found {sentinel})")
    print(f"  Set force=True to rebuild")
else:
    print(f"\n[INFO] Training regression horizon models...")


[INFO] Training regression horizon models...


## 2. Load and Prepare Regression Target

In [27]:
# Check required files
required_files = [
    data_dir / "response_updated.csv",
    features_dir / "train.parquet",
    features_dir / "val.parquet",
    features_dir / "column_step_map.json"
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(
            f"Required file not found: {file}\n"
            f"Please run previous pipeline steps first."
        )

print("[OK] All required files found")

[OK] All required files found


In [28]:
# Load regression targets from response_updated.csv
print("\nLoading regression targets...")
response_df = pd.read_csv(data_dir / "response_updated.csv")
print(f"  Total rows in response_updated.csv: {len(response_df)}")

# Filter to selected Vt type and deduplicate by taking mean of multiple measurements
vt_filtered = response_df[response_df['NAME'] == vt_type][['WAFER_SCRIBE', 'VALUE']]
print(f"  Rows with {vt_type}: {len(vt_filtered)}")
print(f"  Unique wafers: {vt_filtered['WAFER_SCRIBE'].nunique()}")
duplicates = len(vt_filtered) - vt_filtered['WAFER_SCRIBE'].nunique()
if duplicates > 0:
    print(f"  Found {duplicates} duplicate measurements - averaging per wafer")

# Take mean of duplicate measurements per wafer (one row per wafer)
vt_df = vt_filtered.groupby('WAFER_SCRIBE')['VALUE'].mean().reset_index()
vt_df.rename(columns={'VALUE': 'vt_value'}, inplace=True)
print(f"  Final unique wafers: {len(vt_df)}")
print(f"  Vt value range: [{vt_df['vt_value'].min():.4f}, {vt_df['vt_value'].max():.4f}]")
print(f"  Vt value mean: {vt_df['vt_value'].mean():.4f}")
print(f"  Vt value std: {vt_df['vt_value'].std():.4f}")


Loading regression targets...
  Total rows in response_updated.csv: 76644
  Rows with NFET1_VT: 17202
  Unique wafers: 16817
  Found 385 duplicate measurements - averaging per wafer
  Final unique wafers: 16817
  Vt value range: [-0.7109, 1.5781]
  Vt value mean: 0.4976
  Vt value std: 0.1586


In [29]:
# Load existing feature matrices
print("\nLoading feature matrices...")
train_df = pd.read_parquet(features_dir / "train.parquet")
val_df = pd.read_parquet(features_dir / "val.parquet")
print(f"  Train size: {len(train_df)}")
print(f"  Val size: {len(val_df)}")


Loading feature matrices...
  Train size: 13510
  Val size: 3307


In [30]:
# Merge regression target with feature matrices
print("\nMerging targets with features...")
train_size_before = len(train_df)
val_size_before = len(val_df)

train_df = train_df.merge(vt_df, on='WAFER_SCRIBE', how='left')
val_df = val_df.merge(vt_df, on='WAFER_SCRIBE', how='left')

# Verify merge didn't create duplicates
train_size_after = len(train_df)
val_size_after = len(val_df)
if train_size_after != train_size_before or val_size_after != val_size_before:
    raise ValueError(
        f"Merge created duplicates! Train: {train_size_before} -> {train_size_after}, "
        f"Val: {val_size_before} -> {val_size_after}. "
        f"This indicates duplicate WAFER_SCRIBE values in vt_df."
    )
print(f"  Merge successful (no duplicates created)")

# Drop original classification target column
if 'is_outlier' in train_df.columns:
    train_df = train_df.drop(columns=['is_outlier'])
    val_df = val_df.drop(columns=['is_outlier'])

# Check for missing values in target
train_missing = train_df['vt_value'].isna().sum()
val_missing = val_df['vt_value'].isna().sum()
print(f"  Missing targets in train: {train_missing} ({100*train_missing/len(train_df):.2f}%)")
print(f"  Missing targets in val: {val_missing} ({100*val_missing/len(val_df):.2f}%)")

# Drop rows with missing targets
if train_missing > 0 or val_missing > 0:
    train_df = train_df.dropna(subset=['vt_value'])
    val_df = val_df.dropna(subset=['vt_value'])
    print(f"  Dropped {train_missing} train and {val_missing} val rows with missing targets")

print(f"  Final train size: {len(train_df)}")
print(f"  Final val size: {len(val_df)}")


Merging targets with features...
  Merge successful (no duplicates created)
  Missing targets in train: 0 (0.00%)
  Missing targets in val: 0 (0.00%)
  Final train size: 13510
  Final val size: 3307


## 3. Load Column-Step Mapping

In [31]:
# Load column-to-step mapping
print("\nLoading column-step mapping...")
with open(features_dir / "column_step_map.json", 'r') as f:
    column_step_map = json.load(f)

print(f"  Loaded mappings for {len(column_step_map)} features")
print(f"  SeqNo range: {min(column_step_map.values())} to {max(column_step_map.values())}")


Loading column-step mapping...
  Loaded mappings for 10302 features
  SeqNo range: 0 to 240


In [32]:
# Load step sequence to define horizons
print("\nDefining horizons...")
step_seq_file = data_dir / "step_seq.csv"

if not step_seq_file.exists():
    print(f"  WARNING: {step_seq_file} not found.")
    print(f"  Using SeqNo values from column_step_map instead.")
    all_seqnos = sorted(set(column_step_map.values()))
    all_seqnos = [s for s in all_seqnos if s > 0]  # Exclude SeqNo=0
else:
    step_seq_df = pd.read_csv(step_seq_file)
    all_seqnos = sorted(step_seq_df['SeqNo'].unique())
    print(f"  Loaded {len(all_seqnos)} step sequence numbers")


Defining horizons...
  Loaded 240 step sequence numbers


In [33]:
# Define horizons at deciles (10%, 20%, ..., 100% of process)
percentiles = np.percentile(all_seqnos, [10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
horizons = sorted(set([int(p) for p in percentiles]))

print(f"\n  Training models at {len(horizons)} horizons:")
for h in horizons:
    pct = 100 * h / max(all_seqnos)
    print(f"    Horizon {h:>4} ({pct:>5.1f}% complete)")


  Training models at 10 horizons:
    Horizon   24 ( 10.0% complete)
    Horizon   48 ( 20.0% complete)
    Horizon   72 ( 30.0% complete)
    Horizon   96 ( 40.0% complete)
    Horizon  120 ( 50.0% complete)
    Horizon  144 ( 60.0% complete)
    Horizon  168 ( 70.0% complete)
    Horizon  192 ( 80.0% complete)
    Horizon  216 ( 90.0% complete)
    Horizon  240 (100.0% complete)


In [34]:
# Prepare metadata columns
metadata_cols = ['WAFER_SCRIBE', 'LOT_ID', 'vt_value', 'PARAM_END_DATETIME', 
                'first_start_time']
feature_cols = [col for col in train_df.columns if col not in metadata_cols]

print(f"\n  Total available features: {len(feature_cols)}")


  Total available features: 10302


In [35]:
# Extract regression targets
y_train = train_df['vt_value']
y_val = val_df['vt_value']

print(f"\n  Target statistics (train):")
print(f"    Mean: {y_train.mean():.4f}")
print(f"    Std: {y_train.std():.4f}")
print(f"    Range: [{y_train.min():.4f}, {y_train.max():.4f}]")
print(f"\n  Target statistics (val):")
print(f"    Mean: {y_val.mean():.4f}")
print(f"    Std: {y_val.std():.4f}")
print(f"    Range: [{y_val.min():.4f}, {y_val.max():.4f}]")


  Target statistics (train):
    Mean: 0.5051
    Std: 0.1585
    Range: [-0.7109, 1.5781]

  Target statistics (val):
    Mean: 0.4670
    Std: 0.1554
    Range: [-0.5781, 1.3802]


## 4. Multi-Horizon Regression Training Loop

In [36]:
# Train regression models at each horizon
horizon_results = []

print("\n" + "=" * 80)
print("Training regression horizon models...")
print("=" * 80)
print(f"{'Horizon':<10} {'Features':<10} {'RMSE':<12} {'MAE':<12} {'R²':<10}")
print("-" * 80)

for k in tqdm(horizons, desc="Horizons"):
    # Filter features to those available at horizon k
    available_features = [
        col for col in feature_cols 
        if col in column_step_map and column_step_map[col] <= k
    ]
    
    if len(available_features) == 0:
        print(f"\n  WARNING: No features available at horizon {k}, skipping...")
        continue
    
    # Prepare feature matrices
    X_train = train_df[available_features].copy()
    X_val = val_df[available_features].copy()
    
    # Identify categorical features in this subset
    cat_features = []
    for col in available_features:
        is_categorical = (
            X_train[col].dtype == 'object' or
            X_train[col].dtype.name == 'string' or
            str(X_train[col].dtype).startswith('str') or
            col.startswith('LOT_ID') or
            '__EQUIP' in col or
            '__POSITION' in col or
            col in ['first_step', 'last_step']
        )
        if is_categorical:
            cat_features.append(col)
            # Ensure categorical columns are strings
            X_train[col] = X_train[col].fillna('UNKNOWN').astype(str)
            X_val[col] = X_val[col].fillna('UNKNOWN').astype(str)
    
    # Create CatBoost pools
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)
    
    # Train regression model
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function='RMSE',
        eval_metric='RMSE',
        early_stopping_rounds=30,
        random_seed=42,
        verbose=0  # Suppress per-iteration output
    )
    
    model.fit(train_pool, eval_set=val_pool)
    
    # Evaluate
    y_pred = model.predict(X_val)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    # Save model
    model_file = models_dir / f"regression_horizon_{k:03d}_model.cbm"
    model.save_model(str(model_file))
    
    # Store results
    horizon_results.append({
        'horizon': int(k),
        'n_features': len(available_features),
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2)
    })
    
    # Print summary line
    print(f"k={k:<7} {len(available_features):<10} {rmse:<12.6f} {mae:<12.6f} {r2:<10.4f}")

print("-" * 80)


Training regression horizon models...
Horizon    Features   RMSE         MAE          R²        
--------------------------------------------------------------------------------


Horizons:  10%|█         | 1/10 [00:10<01:35, 10.63s/it]

k=24      1766       0.155322     0.114492     0.0005    


Horizons:  20%|██        | 2/10 [00:30<02:07, 15.89s/it]

k=48      4373       0.151736     0.111699     0.0461    


...

k=192     9445       0.132206     0.095689     0.2758    


Horizons:  90%|█████████ | 9/10 [16:27<02:42, 162.35s/it]

k=216     9847       0.132954     0.095125     0.2676    


Horizons: 100%|██████████| 10/10 [19:53<00:00, 119.35s/it]

k=240     10302      0.133566     0.096459     0.2609    
--------------------------------------------------------------------------------


## 5. Save Results and Metrics

In [45]:
# Save results to JSON
results_file = output_dir / "regression_horizon_results.json"
with open(results_file, 'w') as f:
    json.dump({
        'vt_type': vt_type,
        'horizons': horizon_results
    }, f, indent=2)

print(f"\n[OK] Saved regression_horizon_results.json with {len(horizon_results)} entries")


[OK] Saved regression_horizon_results.json with 10 entries


In [46]:
# Summary statistics
print("\n" + "=" * 60)
print("Regression Horizon Model Training Summary")
print("=" * 60)
print(f"Vt type: {vt_type}")
print(f"Total horizons evaluated: {len(horizon_results)}")
print(f"Horizon range: {min(r['horizon'] for r in horizon_results)} to "
      f"{max(r['horizon'] for r in horizon_results)}")
print(f"\nPerformance range:")
print(f"  RMSE: {min(r['rmse'] for r in horizon_results):.6f} to "
      f"{max(r['rmse'] for r in horizon_results):.6f}")
print(f"  MAE:  {min(r['mae'] for r in horizon_results):.6f} to "
      f"{max(r['mae'] for r in horizon_results):.6f}")
print(f"  R²:   {min(r['r2'] for r in horizon_results):.4f} to "
      f"{max(r['r2'] for r in horizon_results):.4f}")

# Find first viable prediction point (R² > 0.6)
viable_horizons = [r for r in horizon_results if r['r2'] > 0.6]
if viable_horizons:
    first_viable = min(viable_horizons, key=lambda r: r['horizon'])
    max_seqno = max(r['horizon'] for r in horizon_results)
    pct_complete = 100 * first_viable['horizon'] / max_seqno
    print(f"\nFirst viable prediction point (R² > 0.6):")
    print(f"  Horizon {first_viable['horizon']} ({pct_complete:.1f}% complete)")
    print(f"  R²: {first_viable['r2']:.4f}")
    print(f"  RMSE: {first_viable['rmse']:.6f}")
    print(f"  MAE: {first_viable['mae']:.6f}")
    print(f"  Features: {first_viable['n_features']}")
else:
    print(f"\nNo horizon achieved R² > 0.6")
    best_horizon = max(horizon_results, key=lambda r: r['r2'])
    print(f"Best R² achieved: {best_horizon['r2']:.4f} at horizon {best_horizon['horizon']}")


Regression Horizon Model Training Summary
Vt type: NFET1_VT
Total horizons evaluated: 10
Horizon range: 24 to 240

Performance range:
  RMSE: 0.132136 to 0.155322
  MAE:  0.095125 to 0.114492
  R²:   0.0005 to 0.2766

No horizon achieved R² > 0.6
Best R² achieved: 0.2766 at horizon 144


## 6. Evaluation Plots (Final Horizon)

In [47]:
# Load best/final model for detailed evaluation
final_horizon = max(r['horizon'] for r in horizon_results)
final_model_file = models_dir / f"regression_horizon_{final_horizon:03d}_model.cbm"

print(f"\n[INFO] Loading final model from horizon {final_horizon} for evaluation...")
from catboost import CatBoostRegressor
final_model = CatBoostRegressor()
final_model.load_model(str(final_model_file))

# Get available features for final horizon
final_available_features = [
    col for col in feature_cols 
    if col in column_step_map and column_step_map[col] <= final_horizon
]

X_val_final = val_df[final_available_features].copy()

# Handle categorical features
cat_features_final = []
for col in final_available_features:
    is_categorical = (
        X_val_final[col].dtype == 'object' or
        X_val_final[col].dtype.name == 'string' or
        str(X_val_final[col].dtype).startswith('str') or
        col.startswith('LOT_ID') or
        '__EQUIP' in col or
        '__POSITION' in col or
        col in ['first_step', 'last_step']
    )
    if is_categorical:
        cat_features_final.append(col)
        X_val_final[col] = X_val_final[col].fillna('UNKNOWN').astype(str)

# Get predictions
y_pred_final = final_model.predict(X_val_final)

print(f"[OK] Predictions generated for {len(y_pred_final)} validation samples")


[INFO] Loading final model from horizon 240 for evaluation...
[OK] Predictions generated for 3307 validation samples


In [48]:
# Plot 1: Predicted vs Actual
print("\n[INFO] Generating Predicted vs Actual plot...")
plt.figure(figsize=(10, 8))
plt.scatter(y_val, y_pred_final, alpha=0.5, s=20)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Vt Value', fontsize=12)
plt.ylabel('Predicted Vt Value', fontsize=12)
plt.title(f'Predicted vs Actual Vt ({vt_type}) - Horizon {final_horizon}\nR² = {r2_score(y_val, y_pred_final):.4f}', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
pred_vs_actual_file = plots_dir / "regression_pred_vs_actual.png"
plt.savefig(pred_vs_actual_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"[OK] Saved to {pred_vs_actual_file}")


[INFO] Generating Predicted vs Actual plot...
[OK] Saved to outputs\plots\regression_pred_vs_actual.png


In [49]:
# Plot 2: Residual Plot
print("\n[INFO] Generating Residual plot...")
residuals = y_val.values - y_pred_final
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_final, residuals, alpha=0.5, s=20)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Vt Value', fontsize=12)
plt.ylabel('Residuals (Actual - Predicted)', fontsize=12)
plt.title(f'Residual Plot ({vt_type}) - Horizon {final_horizon}', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
residual_file = plots_dir / "regression_residuals.png"
plt.savefig(residual_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"[OK] Saved to {residual_file}")


[INFO] Generating Residual plot...
[OK] Saved to outputs\plots\regression_residuals.png


In [50]:
# Plot 3: Feature Importance
print("\n[INFO] Generating Feature Importance plot...")
feature_importance = final_model.get_feature_importance()
feature_names = final_available_features

# Create DataFrame and sort by importance
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(30)

# Assign colors based on feature type
colors = []
for feat in importance_df['feature']:
    if feat.startswith('LOT_ID') or '__EQUIP' in feat or '__POSITION' in feat:
        colors.append('red')  # Lot/categorical features
    elif '__SPC_' in feat:
        colors.append('green')  # SPC features
    else:
        colors.append('blue')  # Sensor features

# Plot
plt.figure(figsize=(12, 10))
plt.barh(range(len(importance_df)), importance_df['importance'], color=colors)
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Feature Importance', fontsize=12)
plt.title(f'Top 30 Features - Regression ({vt_type}) - Horizon {final_horizon}', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
importance_file = plots_dir / "regression_feature_importance.png"
plt.savefig(importance_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"[OK] Saved to {importance_file}")

# Print top 20 features
print("\nTop 20 Most Important Features:")
print("-" * 60)
for i, row in importance_df.head(20).iterrows():
    print(f"{row['feature']:<50} {row['importance']:>8.2f}")
print("-" * 60)


[INFO] Generating Feature Importance plot...
[OK] Saved to outputs\plots\regression_feature_importance.png

Top 20 Most Important Features:
------------------------------------------------------------
PlatenInputTemp__IM0013__MEAN                          4.80
ScannerSuppressionVoltagePre__IM0013__MEAN             2.80
ImplantBeamCurrentFiltered__IM0014__MEAN               2.79
PFGFilamentResistance__IM0013__MEAN                    2.70
IonGauge3Pressure__IM0014__MEAN                        2.33
ESChuckCurrent__IM0013__MEAN                           2.20
ClosedLoopFaradayCurrent__IM0013__MEAN                 1.98
ChuckHeGasFlow__DE0007__MEAN                           1.97
HolderWaterFlow__PV0002__MEAN                          1.34
CC02Concentration1__WP0059__MEAN                       1.32
G031GasBottlePressure__IM0038__MEAN                    1.22
G020GasBottlePressure__IM0014__MEAN                    1.16
UniformCurrent__IM0004__MEAN                           1.07
SharedPressure__DF

In [51]:
# Save evaluation summary
print("\n[INFO] Saving evaluation summary...")
summary_file = output_dir / "regression_evaluation_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("Regression Multi-Horizon Evaluation Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Vt Type: {vt_type}\n")
    f.write(f"Final Horizon: {final_horizon}\n")
    f.write(f"Number of Features: {len(final_available_features)}\n")
    f.write(f"Validation Set Size: {len(y_val)}\n\n")
    
    f.write("Performance Metrics (Final Horizon):\n")
    f.write("-" * 40 + "\n")
    f.write(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred_final)):.6f}\n")
    f.write(f"MAE:  {mean_absolute_error(y_val, y_pred_final):.6f}\n")
    f.write(f"R²:   {r2_score(y_val, y_pred_final):.4f}\n\n")
    
    f.write("Target Statistics:\n")
    f.write("-" * 40 + "\n")
    f.write(f"Mean (actual): {y_val.mean():.6f}\n")
    f.write(f"Std (actual):  {y_val.std():.6f}\n")
    f.write(f"Mean (pred):   {y_pred_final.mean():.6f}\n")
    f.write(f"Std (pred):    {y_pred_final.std():.6f}\n\n")
    
    f.write("Horizon Performance Summary:\n")
    f.write("-" * 60 + "\n")
    f.write(f"{'Horizon':<10} {'Features':<10} {'RMSE':<12} {'MAE':<12} {'R²':<10}\n")
    f.write("-" * 60 + "\n")
    for r in horizon_results:
        f.write(f"{r['horizon']:<10} {r['n_features']:<10} {r['rmse']:<12.6f} {r['mae']:<12.6f} {r['r2']:<10.4f}\n")
    f.write("-" * 60 + "\n")

print(f"[OK] Saved to {summary_file}")


[INFO] Saving evaluation summary...
[OK] Saved to outputs\regression_evaluation_summary.txt


In [52]:
# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Regression horizon model training complete in {elapsed:.2f}s")
print("=" * 60)


[OK] Regression horizon model training complete in 1757.07s
